# 06 — Saving and Exporting

Once you have a CP you like, you'll want to send it somewhere — a plotter, a folder simulator, or a 3D printer. This notebook covers the export formats `eucare` ships with:

- `.heg` — eucare's native YAML serialization.
- SVG — vector for laser cutters / pen plotters.
- A high-level `overlap.save_results` that writes a whole result directory in one call.

STL files can be generated via marching cubes, e.g. for 3D printers. It requires the `[threed]` install extra.

The [FOLD format](https://github.com/edemaine/fold) is currently not supported, but would be nice to have in the future.

In [ ]:
import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import eucare as ec
from eucare import (
    example_graphs,
    example_tilesets,
    rendering,
)
from eucare.rendering import multi_show


In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=2)
G.recompute_lengths_and_angles()

## .heg save

Eucare's native YAML serialization captures the full graph structure plus arbitrary attributes.

Note: the load path currently uses `yaml.SafeLoader`, so graphs with non-trivial Python attributes (tuples, numpy scalars) save but don't round-trip.

In [ ]:
import tempfile, os
from eucare import io

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, 'pattern.heg')
    io.save_graph(path, G)
    size = os.path.getsize(path)
    print(f'wrote {size} bytes')

    # load it back
    G2 = io.load_graph(path)

multi_show(
    [G, G2],
    titles=['original', 'loaded'],
)


## SVG export via `G.save(...)`

`G.save('out')` writes both `out.svg` and `out.png`. `G.show()` displays inline (vector SVG in Jupyter) without writing files. The SVG is the same vector drawing that `CairoRenderer` produced — open it in a browser or Inkscape for the full quality.

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as d:
    out = os.path.join(d, 'pattern')
    G.save(out, **rendering.CREASE_PATTERN_PRESET)
    print('files in temp dir:', sorted(os.listdir(d)))
    print('SVG head:')
    print(open(out + '.svg').read()[:200])

## Plotter-ready SVG via `SvgwriteRenderer`

For laser-cutter / pen-plotter pipelines the dedicated `SvgwriteRenderer` produces an SVG split into `{name}_borders.svg` / `{name}_interior.svg` (so you can use different tool heads for cut vs. score).

## All-in-one with `overlap.save_results`

If you've gone through `fold_complete`, `save_results(result, path)` writes the CP, both folded views, a back-lit composite, and a plotter-ready SVG in one call. See notebook TODO for an example.